# Tranche 1 — executable D2/D3 evidence

Run cells in order using a Python 3 kernel from this repository or its `notebooks/` directory. Only the Python standard library is used. Rust 1.84.1, Stellar CLI 22.8.2, cargo-llvm-cov 0.6.16 and the local-node binaries must already be installed; see [local RPC setup](../docs/local-rpc.md). This notebook does not install tools or reset existing node data.

**Recording:** run P0–P4 before filming; keep the node alive. On camera, run D2-A through D3-C. Full checks and deployment take longer than the two-minute D2 segment. A warm build is labeled as such. All transactions here use isolated standalone funds and fixture assets. No external network transaction is signed.

**Pricing:** Reflector is the only NAV price source. There is no Soroswap TWAP or deviation check, which departs from the original award. A wrong Reflector price that is within the per-feed and vault age limits flows straight into vault accounting. Local tests use a controlled Reflector ABI fixture, not deployed Reflector v6. [D3 acceptance](../docs/d3-acceptance.md) records outstanding full-universe/source/liquidity gates. This notebook does not close them.

Outputs are generated by execution, with dates, revision and artifact hashes. Do not present historical outputs as a fresh run. Logs, node state and new receipt summaries stay in ignored `artifacts/local/` and `.tooling/` directories.

## P0 — repository, pinned tools and evidence helpers

Use the optional repository-local tool installation when present, otherwise the tools on PATH. The notebook records the Git revision and working-tree status; changes after the recorded commit remain visible.

In [1]:
import hashlib
import json
import os
from pathlib import Path
import shlex
import socket
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from decimal import Decimal
import urllib.request

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/local-smoke.py").is_file())
ENV = os.environ.copy()
if (ROOT / ".tooling/cargo/bin/cargo").exists():
    ENV.update(RUSTUP_HOME=str(ROOT / ".tooling/rustup"),
               CARGO_HOME=str(ROOT / ".tooling/cargo"))
    ENV["PATH"] = f"{ROOT}/.tooling/bin:{ROOT}/.tooling/cargo/bin:" + ENV["PATH"]
OUT = ROOT / "artifacts/local/tranche-1-demo"
OUT.mkdir(parents=True, exist_ok=True)
RUN_AT = datetime.now(timezone.utc).isoformat()
RPC = "http://127.0.0.1:8003"
PASSPHRASE = "Standalone Network ; September 2026"

def run(args, label, tail=24):
    print("$", shlex.join(["python3" if a == sys.executable else a for a in args]), flush=True)
    started = time.monotonic()
    result = subprocess.run(args, cwd=ROOT, env=ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (OUT / (label + ".log")).write_text(result.stdout)
    print("\n".join(result.stdout.replace(str(ROOT), ".").splitlines()[-tail:]))
    print(f"Exit {result.returncode}; {time.monotonic() - started:.1f}s; log: artifacts/local/tranche-1-demo/{label}.log")
    result.check_returncode()
    return result.stdout

def rpc(method, **params):
    request = urllib.request.Request(RPC, data=json.dumps({
        "jsonrpc": "2.0", "id": 1, "method": method, **({"params": params} if params else {})
    }).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=20) as response:
        payload = json.load(response)
    assert "error" not in payload, payload
    return payload["result"]

def invoke(contract, method, args=(), send="no"):
    # Every invocation rechecks the network before using a local fixture identity.
    assert rpc("getNetwork")["passphrase"] == PASSPHRASE
    result = subprocess.run([
        "stellar", "contract", "invoke", "--network", "local",
        "--source", "local-root", "--id", contract, "--send=" + send,
        "--", method, *args
    ], cwd=ROOT, env=ENV, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr)
    return json.loads(result.stdout) if result.stdout.strip() else None

def test(package, name):
    output = run(["cargo", "test", "--locked", "-p", package,
                  "tests::" + name, "--", "--exact", "--nocapture"], name, tail=16)
    assert "1 passed; 0 failed" in output, "The exact named test did not run"

print("Run UTC:", RUN_AT)
REVISION = run(["git", "rev-parse", "HEAD"], "revision").strip()
run(["git", "status", "--short"], "worktree")
assert "rustc 1.84.1 " in run(["rustc", "--version"], "rust")
assert "stellar 22.8.2 " in run(["stellar", "--version"], "stellar")
assert "0.6.16" in run(["cargo", "llvm-cov", "--version"], "coverage-tool")

Run UTC: 2026-11-09T16:39:39.991265+00:00
$ git rev-parse HEAD
e61cb9241397a3e6bd6181c00d5bb1d091c518f3
Exit 0; 0.0s; log: artifacts/local/tranche-1-demo/revision.log
$ git status --short
 M notebooks/tranche-1-demo.ipynb
Exit 0; 0.0s; log: artifacts/local/tranche-1-demo/worktree.log
$ rustc --version


rustc 1.84.1 (e71f9a9a9 2025-01-27)
Exit 0; 0.2s; log: artifacts/local/tranche-1-demo/rust.log
$ stellar --version
stellar 22.8.2 (7ab5f0565ed3a03b41789aaac7211c286fa0028c)
stellar-xdr 22.1.0 (e13922970800d95b523413018b2279df42df3442)
xdr curr (529d5176f24c73eeccfa5eba481d4e89c19b1181)
Exit 0; 0.1s; log: artifacts/local/tranche-1-demo/stellar.log
$ cargo llvm-cov --version
cargo-llvm-cov 0.6.16
Exit 0; 0.1s; log: artifacts/local/tranche-1-demo/coverage-tool.log


## P1 — complete build, tests and coverage (before recording)

This is the authoritative repository check: optimized upstream/Etesia builds, fmt, strict clippy, all native tests and a ≥90% production line-coverage assertion. Existing upstream DeFindex and LLVM macro warnings may appear; do not describe the entire dependency output as warning-free. The full log is saved.

In [2]:
check_output = run(["sh", "scripts/check.sh"], "full-check", tail=45)
CHECK_FINISHED_AT = datetime.now(timezone.utc).isoformat()
print("Native suite summaries:")
for line in check_output.splitlines():
    if "test result:" in line:
        print(line)

$ sh scripts/check.sh
test tests::recovery_rejects_deleveraging_that_worsens_pool_health ... ok
test tests::reward_claim_measures_custody_and_rejects_false_report_atomically ... ok
test tests::recovery_rejects_less_debt_when_losses_increase_leverage ... ok
test tests::stale_price_basket_waives_only_exiting_entitlement ... ok
test tests::malicious_output_and_reentry_roll_back_nonce_fees_and_transfers ... ok
test tests::reward_assets_cannot_be_bought_but_can_be_realized ... ok
test tests::reject_expired_changed_or_unbounded_plans_before_custody_changes ... ok
test tests::recovery_allows_partial_deleveraging_while_limits_remain_breached ... ok
test tests::target_auth_guarded_swap_and_nonce_replay ... ok
test tests::price_validity_insolvency_and_liquidity_views ... ok
test tests::transfers_allowances_expiration_and_delegated_exit ... ok
test tests::repeated_individually_legal_public_unwinds_obey_aggregate_turnover ... ok
test tests::nonprofitable_fee_settlement_never_resets_mark_and_fracti

## P2 — start a fresh isolated node (before recording)

This starts a background process owned by the notebook, waits for readiness and uses a new data directory. Ports must be free. Run the cleanup cell when finished; do not restart the kernel before stopping the node. If the kernel is interrupted during startup, run cleanup. Setup can take several minutes.

In [3]:
for port in (8003, 11625, 11626, 11825, 11826, 1570):
    with socket.socket() as sock:
        assert sock.connect_ex(("127.0.0.1", port)) != 0, f"Port {port} is already in use; stop that node first"
(ROOT / ".tooling").mkdir(exist_ok=True)
NODE_DATA = Path(tempfile.mkdtemp(prefix="tranche-1-", dir=ROOT / ".tooling"))
with (OUT / "node.log").open("w") as stream:
    node = subprocess.Popen([sys.executable, "scripts/local-node.py", str(NODE_DATA)],
                            cwd=ROOT, env=ENV, stdout=stream, stderr=subprocess.STDOUT)
try:
    deadline = time.monotonic() + 240
    while "Local RPC ready" not in (OUT / "node.log").read_text():
        assert node.poll() is None, (OUT / "node.log").read_text()
        assert time.monotonic() < deadline, "Node startup timed out; inspect artifacts/local/tranche-1-demo/node.log"
        time.sleep(1)
    assert rpc("getNetwork")["passphrase"] == PASSPHRASE
    assert rpc("getHealth")["status"] == "healthy"
    print("Healthy isolated RPC:", RPC)
    print("Network:", PASSPHRASE)
except BaseException:
    node.terminate()
    node.wait(timeout=60)
    raise

Healthy isolated RPC: http://127.0.0.1:8003
Network: Standalone Network ; September 2026


## P3 — fresh D2 deployment and interaction (before recording)

Deploys six fixture assets and the vault, deposits 100 USDC, publishes a target, performs a guarded swap, collects fees, unwinds and redeems all holder shares. The existing smoke script asserts the holder's final balance is zero. This is the deployment segment to record separately if showing a time-compressed run.

In [4]:
run([sys.executable, "scripts/local-smoke.py"], "d2-local-deployment")
d2 = json.loads((ROOT / "artifacts/local/manifest.json").read_text())
assert d2["result"] == "passed" and d2["passphrase"] == PASSPHRASE
assert len(d2["assets"]) == 6
print("Deployed vault:", d2["vault"])
print("Deposit USDC:", Decimal(d2["deposited_assets"]) / 10**7)
print("Minted shares:", Decimal(d2["minted_shares"]) / 10**12)
print("Redeemed USDC:", Decimal(d2["redeemed_assets"]) / 10**7)

$ python3 scripts/local-smoke.py
funded USDC CAGNAU6CL552TNILAUT64ASQD6XQI6DRJ5L6QIVBDD3TKFPGGKALFJOY
funded XLM CDIOCTM3OU3EVHN65OFMSLIV6EAFJJFDMXX3C66A66KWLYZWF4VOFVA6
funded AQUA CDQ44S3GUVKLUAZZTVZ2EE5G2U2WC4OPALDQ2GVR2ZEBYM43K7I7YEKI
funded ETH CCYFXLCP4Y7V2DFIKYZ2HRLCCMK4PRK3Z5DNWGVRNP2R7FYJXU6PAYBR
funded BTC CAKL2VH4A2BUOYO7W4DRSUL67SBQLAP4BDUBEZYQPCF6KKKK5Q7VYX63
funded USTRY CC6VGB2XFGHT2QT237EHZLGMTLHSNY4KTIT2SCKBEMC53MFIGE2Y2YSL
deposited 100000000063419
guarded swap confirmed
local RPC round trip passed 999999997
Exit 0; 41.9s; log: artifacts/local/tranche-1-demo/d2-local-deployment.log
Deployed vault: CAYI6ELEA2FKIJBQJNUZNKKON2AFNAYYYI6JMUZMKI4FWDM57YMVZS56
Deposit USDC: 100
Minted shares: 100.000000063419
Redeemed USDC: 99.9999997


## P4 — prepare the D3 pricing provider and local NAV round trip

The local smoke deploys local tokens and a controlled Reflector-interface feed, verifies 1,005 USDC NAV, deposits, rejects an invalid-feed redemption, restores the feed and redeems. Run before filming. Its fixture marks are not current mainnet quotes.

In [5]:
run([sys.executable, "scripts/d3-local-smoke.py"], "d3-local-deployment")
d3 = json.loads((ROOT / "artifacts/local/d3/manifest.json").read_text())
assert d3["result"] == "passed" and d3["nav"] == 10_050_000_000
print("Local NAV:", Decimal(d3["nav"]) / 10**7, "USDC")
print("Pricing provider:", d3["provider"])
print("Feed:", d3["feed"])

$ python3 scripts/d3-local-smoke.py
funded LOCAL USDC
funded LOCAL XLM
funded LOCAL AQUA
funded LOCAL ETH
funded LOCAL BTC
funded LOCAL USTRY
feed ready XLM
feed ready AQUA
feed ready ETH
feed ready BTC
feed ready USTRY
D3 local RPC passed 10050000000 999999998
Exit 0; 45.0s; log: artifacts/local/tranche-1-demo/d3-local-deployment.log
Local NAV: 1005 USDC
Pricing provider: CAT4BCVE6XQ5HVOICMR3TGYVH4UA4A33YLDCMIT5V5ZXXS7P5KN7VHK7
Feed: controlled local ABI fixture; deployed Reflector v6 spec is incompatible with pinned CLI 22


## D2-A — warm compilation

Here I’m compiling the Etesia vault contracts into Soroban WebAssembly. The dependencies and full validation checks were prepared earlier, so this repeat build uses the existing cache. The output shows the optimized vault file, its size and its SHA-256 hash, which identifies the exact artifact used in the deployment.

In [6]:
run(["cargo", "build", "--workspace", "--locked", "--release",
     "--target", "wasm32-unknown-unknown"], "warm-build", tail=8)
run(["stellar", "contract", "optimize", "--wasm",
     "target/wasm32-unknown-unknown/release/etesia_vault.wasm"], "optimize-vault", tail=8)
wasm = ROOT / "target/wasm32-unknown-unknown/release/etesia_vault.optimized.wasm"
print(wasm.name, "bytes:", wasm.stat().st_size)
print("SHA-256:", hashlib.sha256(wasm.read_bytes()).hexdigest())

$ cargo build --workspace --locked --release --target wasm32-unknown-unknown
    Finished `release` profile [optimized] target(s) in 0.12s
Exit 0; 0.2s; log: artifacts/local/tranche-1-demo/warm-build.log
$ stellar contract optimize --wasm target/wasm32-unknown-unknown/release/etesia_vault.wasm
Reading: target/wasm32-unknown-unknown/release/etesia_vault.wasm (126472 bytes)
Optimized: target/wasm32-unknown-unknown/release/etesia_vault.optimized.wasm (98311 bytes)
Exit 0; 0.4s; log: artifacts/local/tranche-1-demo/optimize-vault.log
etesia_vault.optimized.wasm bytes: 98311
SHA-256: 039c0ef0194bd7fa4baab4567d234571182e410a9789b1c70911b5138d6fdc67


## D2-B — measured coverage

This is the production line-coverage report from the full validation run. It covers the vault, adapter and pricing provider, with a required minimum of ninety percent. Tests, fixture tools, vendored dependencies and generated wire declarations are excluded. The displayed result records the measured coverage, test summaries, source revision and completion time.

In [7]:
lines = json.loads((ROOT / "coverage/coverage.json").read_text())["data"][0]["totals"]["lines"]
assert lines["percent"] >= 90
print("Full checks completed UTC:", CHECK_FINISHED_AT)
print("Source revision:", REVISION)
print(f"Production line coverage: {lines['covered']}/{lines['count']} = {lines['percent']:.2f}% (required: 90%)")
for line in check_output.splitlines():
    if "test result:" in line:
        print(line)

Full checks completed UTC: 2026-11-09T16:40:18.074248+00:00
Source revision: e61cb9241397a3e6bd6181c00d5bb1d091c518f3
Production line coverage: 2003/2134 = 93.86% (required: 90%)
test result: ok. 3 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.50s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
test result: ok. 12 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.11s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
test result: ok. 43 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 1.73s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filte

## D2-C — share correctness and rejection of violations

I’m running four focused tests. They check deposit, mint, withdraw and redeem accounting, reject unauthorized share issuance, reject invalid rebalance plans and prevent nonce replay. Each command selects one exact test, so “one passed, forty-three filtered out” means that test passed and the other vault tests were not selected. The full suite ran during preparation.

In [8]:
test("etesia-vault", "funded_seed_and_all_standard_round_trips")
test("etesia-vault", "exact_deposit_auth_tree_and_no_unauthorized_issuance")
test("etesia-vault", "reject_expired_changed_or_unbounded_plans_before_custody_changes")
test("etesia-vault", "target_auth_guarded_swap_and_nonce_replay")

$ cargo test --locked -p etesia-vault tests::funded_seed_and_all_standard_round_trips -- --exact --nocapture
   Compiling etesia-vault v0.1.0 (./contracts/vault)
    Finished `test` profile [unoptimized] target(s) in 6.39s
     Running unittests src/lib.rs (target/debug/deps/etesia_vault-1283b7f722122507)

running 1 test
Writing test snapshot file for test "tests::funded_seed_and_all_standard_round_trips" to "test_snapshots/tests/funded_seed_and_all_standard_round_trips.1.json".
test tests::funded_seed_and_all_standard_round_trips ... ok

test result: ok. 1 passed; 0 failed; 0 ignored; 0 measured; 42 filtered out; finished in 0.13s

Exit 0; 6.6s; log: artifacts/local/tranche-1-demo/funded_seed_and_all_standard_round_trips.log
$ cargo test --locked -p etesia-vault tests::exact_deposit_auth_tree_and_no_unauthorized_issuance -- --exact --nocapture
    Finished `test` profile [unoptimized] target(s) in 0.11s
     Running unittests src/lib.rs (target/debug/deps/etesia_vault-1283b7f722122507

## D2-D — re-query local transaction receipts

The preparation run deployed the vault, deposited 100 USDC, issued shares, executed a guarded action and redeemed the holder’s position. I’m now querying the local RPC to verify the recorded D2 and D3 transaction receipts and checking that the smart contract hashes match our build artifacts. These transactions use test assets on an isolated local network.

In [9]:
assert rpc("getNetwork")["passphrase"] == PASSPHRASE
print("Isolated network:", PASSPHRASE)
verified = {}
for label, manifest in [("D2", d2), ("D3", d3)]:
    assert manifest["transactions"], "No transaction receipts captured"
    receipts = []
    for item in manifest["transactions"]:
        receipt = rpc("getTransaction", hash=item["hash"])
        assert receipt["status"] == "SUCCESS", receipt
        receipts.append({**item, "status": receipt["status"], "ledger": receipt["ledger"]})
    for name, digest in manifest["wasm_sha256"].items():
        artifact = ROOT / "target/wasm32-unknown-unknown/release" / name
        assert hashlib.sha256(artifact.read_bytes()).hexdigest() == digest, name
    verified[label] = {"vault": manifest["vault"], "receipts": receipts,
                       "wasm_sha256": manifest["wasm_sha256"]}
    print(label, manifest["vault"], len(receipts), "SUCCESS receipts; WASM hashes match")
    for item in receipts:
        if item["method"] in ("deposit", "execute", "redeem"):
            print(item["method"], item["status"], "ledger", item["ledger"], item["hash"])
(OUT / "verified-receipts.json").write_text(json.dumps({
    "verified_at": datetime.now(timezone.utc).isoformat(), "revision": REVISION,
    "network": PASSPHRASE, "scope": "local fixture transactions only", "scenarios": verified
}, indent=2) + "\n")

Isolated network: Standalone Network ; September 2026
D2 CAYI6ELEA2FKIJBQJNUZNKKON2AFNAYYYI6JMUZMKI4FWDM57YMVZS56 26 SUCCESS receipts; WASM hashes match
deposit SUCCESS ledger 57 e4bb558c9854b9fa62bb45af3722ca5588de997b5439d7f9080b720af438ec13
execute SUCCESS ledger 60 fa08d01f71419895da06b3e4d8a6e4a8e617deac9d6c27c6a4aea39a2c56cf0b
redeem SUCCESS ledger 65 c722b5dc9e401a4d4c5249f265d3c57ece6156191addc5e38ccaac72afd8310d
D3 CAURIK2T3BA3XPSGEZQ6NB4RFWAB3KPL4ILGGD742SKGWYWEPAE6KULN 29 SUCCESS receipts; WASM hashes match
deposit SUCCESS ledger 105 4cc6187ffdb1a4e2759999ead4c3f398e96e15e6389ce30224d012d61ae776c2
redeem SUCCESS ledger 110 baa98ae4bace06443e39279d8fad7ee856575395a0a1ac4a97b05b876fe32662


12152

## D3-A — read the Reflector mark

This demonstration uses a controlled Reflector-interface feed with local test assets. The provider checks the feed's identity, price and age, then expresses the price in USDC per asset. There is no second price source or deviation check.

In [10]:
asset = next(a["address"] for a in d3["assets"] if a["symbol"] == "XLM")
source = next(s for s in d3["provider_config"]["sources"] if s["asset"] == asset)
feed = source["numerator"]["contract"]
provider = d3["provider"]
assert invoke(provider, "config_hash") == d3["config_hash"]

def set_fixture_price(price, age=0):
    invoke(feed, "update", ["--asset", json.dumps({"Stellar": asset}),
                           "--price", str(price), "--timestamp", str(int(time.time()) - age)], send="yes")

set_fixture_price(10**14)
diagnostic = invoke(provider, "diagnose", ["--asset", asset])
assert diagnostic["error"] == 0, diagnostic
(reading,) = diagnostic["numerator"]
mark = invoke(provider, "mark", ["--asset", asset])
assert mark == diagnostic["mark"] and mark["timestamp"] == reading["timestamp"], mark
print("LOCAL controlled feed; XLM fixture")
print("Feed reading (timestamp, price):", (reading["timestamp"], reading["price"]))
print("Reflector mark:", Decimal(int(mark["price"]))/10**12, "USDC/asset")
print("Feed age limit:", source["numerator"]["max_age"], "s")

LOCAL controlled feed; XLM fixture
Feed reading (timestamp, price): (1794242536, '100000000000000')
Reflector mark: 1 USDC/asset
Feed age limit: 3600 s


## D3-B — reject a stale price and restore it

I’m backdating the controlled feed price to 60 seconds past the configured 3600-second age limit. The provider rejects it with a stale-price error and returns no mark. Restoring a fresh price makes the Reflector mark available again. Age is the only check on a Reflector price: a wrong price that is still fresh would be accepted.

In [11]:
try:
    set_fixture_price(10**14, age=source["numerator"]["max_age"] + 60)
    rejected = invoke(provider, "diagnose", ["--asset", asset])
    assert rejected["error"] == 8 and int(rejected["mark"]["price"]) == 0, rejected
    print("Diagnostic error:", rejected["error"], "Stale; mark price:", rejected["mark"]["price"])
    try:
        invoke(provider, "mark", ["--asset", asset])
    except RuntimeError as error:
        assert "#8" in str(error), str(error)
        print("mark rejected with contract error #8")
    else:
        raise AssertionError("Stale price was accepted")
finally:
    set_fixture_price(10**14)
assert int(invoke(provider, "mark", ["--asset", asset])["price"]) == 10**12
print("Fresh price restored; Reflector mark available again")

Diagnostic error: 8 Stale; mark price: 0
mark rejected with contract error #8
Fresh price restored; Reflector mark available again


## D3-C — NAV, shares and atomic refusal

This contract test checks a 1000-USDC seed, a 1-USDC deposit and a donated asset worth 100 USDC, giving a NAV of 1101 USDC. It also verifies that unavailable pricing blocks deposits and redemption without changing state, holdings, shares or events. Valid pricing allows redemption; stale pricing blocks valuation.

In [12]:
test("etesia-pricing", "new_vault_uses_provider_and_failures_preserve_state")
print("Native contract assertion: 1,000 seed + 1 deposit + 100 asset value = 1,101 USDC NAV")
print("Separate local RPC assertion from P4:", Decimal(d3["nav"])/10**7, "USDC")

$ cargo test --locked -p etesia-pricing tests::new_vault_uses_provider_and_failures_preserve_state -- --exact --nocapture
   Compiling etesia-vault v0.1.0 (./contracts/vault)
   Compiling etesia-pricing v0.1.0 (./contracts/pricing)
    Finished `test` profile [unoptimized] target(s) in 8.34s
     Running unittests src/lib.rs (target/debug/deps/etesia_pricing-42e9f6379fc4a01d)

running 1 test
Writing test snapshot file for test "tests::new_vault_uses_provider_and_failures_preserve_state" to "test_snapshots/tests/new_vault_uses_provider_and_failures_preserve_state.1.json".
test tests::new_vault_uses_provider_and_failures_preserve_state ... ok

test result: ok. 1 passed; 0 failed; 0 ignored; 0 measured; 11 filtered out; finished in 0.09s

Exit 0; 8.5s; log: artifacts/local/tranche-1-demo/new_vault_uses_provider_and_failures_preserve_state.log
Native contract assertion: 1,000 seed + 1 deposit + 100 asset value = 1,101 USDC NAV
Separate local RPC assertion from P4: 1005 USDC


## Cleanup — after recording

Stop only the process started by this notebook. Keep logs and node data for review. Public submission still needs the uploaded video, a public source revision and resolution/disclosure of the acceptance gaps in the repository's D1/D3 records.

In [13]:
if "node" in globals() and node.poll() is None:
    node.terminate()
    node.wait(timeout=60)
    print("Notebook-owned local node stopped; artifacts retained")

Notebook-owned local node stopped; artifacts retained
